<a href="https://colab.research.google.com/github/esero88/eser-portfolio/blob/main/pension-retention-analytics/notebooks/01_egm_veri_toplama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EGM Veri Toplama — BES Kalıcılık ve Rekabet Projesi

Emeklilik Gözetim Merkezi'nin (egm.org.tr) yayınladığı BES istatistik raporlarını
tarih tarih indirir. Her rapor tek bir tarihin anlık görüntüsüdür; zaman serisi
için tarih listesindeki her tarih tek tek geziliyor.

**Hedef raporlar**

| Kod | Rapor | Ne için |
|---|---|---|
| `GR-01-015` | Sözleşmelerin Yürürlüğe Giriş Ayına Göre Sonlanma Oranı | Kohort kalıcılık eğrisi (2003.10 →) |
| `GR-01-010` | Sonlanan Sözleşmelerin Sonlanma Nedenine Göre Dağılımı | Ayrılma nedenleri kırılımı |
| `GR-01-019` | Cayma Hakkı Kullanım Oranı | Cayma trendi |
| `EGM-05-006` | Emeklilik Şirketi ve Faiz Detayında Fon Büyüklükleri | Şirket bazında pazar payı |
| `EGM-05-001` | Fon Bazında Fon İşletim Gideri Kesinti Oranları | Maliyet rekabeti |

**Akış:** Kurulum → Script → Keşif (`--probe`) → Küçük deneme → Tam toplama → İndir

---

> **Kaynak:** Bu defterde toplanan tüm istatistiksel veriler Emeklilik Gözetim
> Merkezi A.Ş.'ye (EGM) aittir. Kaynak gösterilmeden alıntı yapılamaz.

## 1. Kurulum

Playwright ve Chromium kuruluyor. EGM sayfası JavaScript ile render edildiği için
`requests` boş tablo döndürür; gerçek bir tarayıcı motoru gerekiyor.

Ardından Google Drive bağlanıyor. İnen dosyalar doğrudan Drive'a yazılıyor —
Colab'ın geçici diski oturum kapanınca siliniyor, Drive'daki kalıyor. Toplama
yarıda kesilirse tekrar çalıştırmak yeterli; inmiş dosyalar atlanıyor.

Colab oturumu sıfırlandığında bu iki hücre tekrar çalıştırılmalı.

In [ ]:
!pip install playwright -q
!playwright install chromium
!playwright install-deps chromium

Installing dependencies...
Hit:1 http://security.ubuntu.com/ubuntu noble-security InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Fetched 3,917 B in 1s (4,694 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libasound2t64 is already the newest version (1.2.11-1ubuntu0.3).
libatk-bridge2.0-0t64 is already t

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

HEDEF = "/content/drive/MyDrive/egm-bes-projesi/data/raw"
!mkdir -p "$HEDEF"
print("Hedef klasör:", HEDEF)

## 2. Toplayıcı script'i

Script repo'da (`scripts/egm_collector.py`) duruyor; buraya kopyalanmak yerine
oradan çekiliyor. Tek doğru nüsha repo olduğu için defterdeki kopya eskimiyor.

Ayrı bir dosya olarak çalıştırılmasının nedeni: Playwright'ın sync API'si
notebook'un asyncio döngüsüyle çakışıyor. `!python` ile ayrı süreç olarak
çalıştırıldığında sorun kalmıyor.

In [ ]:
!wget -q -O egm_collector.py https://raw.githubusercontent.com/esero88/eser-portfolio/main/pension-retention-analytics/scripts/egm_collector.py
!wc -l egm_collector.py

368 egm_collector.py


## 3. Keşif

`--probe` hiçbir şey indirmez; sayfayı açıp iki şeyi raporlar:

- Tarih listesi bulundu mu, kaç tarih var
- "Excel'e Aktar" düğmesi görülebiliyor mu

`RAPOR_URL` değişkenine raporun EGM'deki adresi yazılıyor.

In [ ]:
RAPOR_URL = "https://egm.org.tr/bilgi-merkezi/istatistikler?r=b5eea773-1617-4344-8e66-1771226ee845"   # <-- raporun linkini buraya yapıştır
RAPOR_KODU = "GR-01-015"

assert RAPOR_URL, "Önce RAPOR_URL'i doldur"
!python egm_collector.py --url "$RAPOR_URL" --code "$RAPOR_KODU" --probe

Aciliyor: https://egm.org.tr/bilgi-merkezi/istatistikler?r=b5eea773-1617-4344-8e66-1771226ee845
83 tarih bulundu.  Ilk: '29 Eylül 2026 Salı'  Son: '30 Aralık 2018 Pazar'
Excel dugmesi: BULUNDU

--- ilk 10 tarih ---
   29 Eylül 2026 Salı -> 2026-09-29
   30 Ağustos 2026 Pazar -> 2026-08-30
   30 Temmuz 2026 Perşembe -> 2026-07-30
   29 Haziran 2026 Pazartesi -> 2026-06-29
   30 Mayıs 2026 Cumartesi -> 2026-05-30
   29 Nisan 2026 Çarşamba -> 2026-04-29
   30 Mart 2026 Pazartesi -> 2026-03-30
   27 Şubat 2026 Cuma -> 2026-02-27
   30 Ocak 2026 Cuma -> 2026-01-30
   30 Aralık 2025 Salı -> 2025-12-30

Kesif modu, indirme yapilmadi.


### Çıktının okunması

| Çıktı | Anlamı |
|---|---|
| "N tarih bulundu" + "Excel düğmesi: BULUNDU" | Yolunda, toplamaya geçilebilir |
| "tarih listesi bulunamadı" | Tarih seçici normal bir `<select>` değil; `tarih_secicisini_bul()` güncellenmeli |
| "Excel düğmesi: BULUNAMADI" | `export_dugmesini_bul()` güncellenmeli |
| Zaman aşımı / bağlantı hatası | EGM, Colab'ın IP'sini engelliyor olabilir; script yerel makinede çalıştırılmalı |

## 4. Küçük deneme

Tam toplamadan önce üç tarihle sınanıyor: dosyalar iniyor mu, içerikleri dolu mu.

In [ ]:
!python egm_collector.py --url "$RAPOR_URL" --code "$RAPOR_KODU" --out "$HEDEF" --limit 3

In [ ]:
import pandas as pd
from pathlib import Path

dosyalar = sorted(Path(HEDEF, RAPOR_KODU).glob("*.xlsx"))
print(f"{len(dosyalar)} dosya indi:")
for d in dosyalar[:5]:
    print(" ", d.name, f"({d.stat().st_size:,} byte)")
if len(dosyalar) > 5:
    print(f"  ... ve {len(dosyalar) - 5} dosya daha")

if dosyalar:
    ornek = pd.read_excel(dosyalar[-1], dtype=str)
    print(f"\n{dosyalar[-1].name}  ->  {ornek.shape[0]} satır × {ornek.shape[1]} kolon")
    display(ornek.head(10))

## 5. Tam toplama

Tarih listesinin tamamı indiriliyor. Zaten inmiş dosyalar atlanıyor, yarıda
kesilirse tekrar çalıştırıldığında kaldığı yerden devam ediyor.

`--delay 2.5` istekler arasındaki bekleme; siteyi yormamak için bırakılmıştır.

In [ ]:
!python egm_collector.py --url "$RAPOR_URL" --code "$RAPOR_KODU" --out "$HEDEF" --delay 2.5

## 6. Diğer raporlar

Her raporun kendi adresi var. Aynı işlem, rapor kodu ve link değiştirilerek
tekrarlanıyor.

In [ ]:
# Her rapor için: linki zincir ikonundan ("Rapor adresini kopyala") al,
# kodu sayfadaki gri yazıdan teyit et.
RAPOR_URL_2  = ""             # <-- raporun linki
RAPOR_KODU_2 = "EGM-05-006"   # <-- raporun kodu

if RAPOR_URL_2:
    !python egm_collector.py --url "$RAPOR_URL_2" --code "$RAPOR_KODU_2" --out "$HEDEF" --delay 2.5
else:
    print("RAPOR_URL_2 boş — doldurulunca çalışır.")

## 7. Kontrol

Dosyalar Drive'da; indirmeye gerek yok. Bu hücre hangi rapordan kaç dosya
toplandığını gösterir.

In [ ]:
from pathlib import Path

kok = Path(HEDEF)
if not kok.exists():
    print("Henüz dosya yok.")
else:
    toplam_dosya = 0
    for klasor in sorted(k for k in kok.iterdir() if k.is_dir()):
        dosyalar = list(klasor.glob("*.xlsx"))
        boyut = sum(d.stat().st_size for d in dosyalar) / 1024
        toplam_dosya += len(dosyalar)
        print(f"{klasor.name:<14} {len(dosyalar):>4} dosya   {boyut:>8,.0f} KB")
    print(f"\nToplam: {toplam_dosya} dosya")

---

## Sonraki adım

Ham dosyalar toplandıktan sonra:

1. **Temizlik** — iki satıra bölünmüş başlıkların düzleştirilmesi, bilimsel
   gösterimdeki sayıların düzeltilmesi, uzun formata (`rapor_tarihi, şirket,
   segment, metrik, değer`) çevrilmesi.
2. **Ambar** — temiz tabloların PostgreSQL'e yüklenmesi, yıldız şema.
3. **Dashboard** — Tableau: kohort kalıcılık eğrisi, sonlanma nedenleri,
   pazar payı, maliyet karşılaştırması.
4. **Agent** — metrik sözlüğü, doğal dilden SQL, 30 soruluk doğruluk testi.